# Common Preprocessing — Kosovo Investment Analysis

Data from ATK investment reports, 2019–2026. The last year covers January–June only. Each row represents an aggregated group, not an individual taxpayer.

**Run first:** `python -m phase_1.run_common_preprocessing` from the project root. This notebook reviews the saved result; it does not re-run or modify the preprocessing pipeline.

## 1. Load the processed data

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/schema.py").exists()), None)
if root is None:
    raise FileNotFoundError("Open the notebook inside the project folder.")
sys.path.insert(0, str(root))

from src.schema import SOURCE_COLUMNS, MONEY, CATEGORIES, BUSINESS_KEY
from src.io_utils import load_dataset

path = root / "data/processed/common_dataset.csv"
if not path.exists():
    raise FileNotFoundError("Run: python -m phase_1.run_common_preprocessing")
df = load_dataset(path)
print(f"Records: {len(df):,} | Source attributes: {len(SOURCE_COLUMNS)}")
display(df[SOURCE_COLUMNS].head(5))

## 2. Structure and attribute types

In [ ]:
df[SOURCE_COLUMNS].info()
display(df[SOURCE_COLUMNS].describe(include="all").T.round(2))

`year` and `month` identify the reporting period; sector, registration status and municipality are categorical. `taxpayer_count` is a count, while the six VAT-investment columns are numerical amounts. The six amounts should not automatically be added together.

## 3. Reporting coverage and integration

In [ ]:
year_counts = df.groupby("year", observed=True).size().rename("records")
display(year_counts.rename_axis("year").reset_index())
periods = df.groupby("year", observed=True)["month"].nunique().rename("months")
display(periods.rename_axis("year").reset_index())
print("Original files:", df["source_file"].nunique())

The annual Excel reports were standardized and combined by the processing scripts. The coverage for 2026 is partial, so full-year comparisons should exclude it or compare January–June across years.

## 4. Data quality: missing values and duplicates

In [ ]:
nulls = df[SOURCE_COLUMNS].isna().sum().rename("nulls").reset_index()
nulls.columns = ["attribute", "nulls"]
display(nulls)

unclassified = df["primary_sector"].eq("Mungon aktiviteti")
print(f"Unclassified sector: {unclassified.sum():,} ({unclassified.mean():.2%})")
print("Duplicate source rows:", int(df.duplicated(subset=["source_file", "source_sheet", "source_excel_row"]).sum()))
print("Duplicate group keys:", int(df.duplicated(subset=BUSINESS_KEY).sum()))

No literal nulls were found in the 12 source attributes. `Mungon aktiviteti` is an **unclassified sector**, not a blank cell: those records are retained and flagged rather than assigned an invented sector.

## 5. Data quality: zeros and negative amounts

In [ ]:
money_summary = pd.DataFrame({
    "zeros": df[MONEY].eq(0).sum(),
    "negative_cells": df[MONEY].lt(0).sum(),
}).reset_index().rename(columns={"index": "attribute"})
display(money_summary)

any_negative = df[MONEY].lt(0).any(axis=1)
all_zero = df[MONEY].eq(0).all(axis=1)
print("Rows with at least one negative:", int(any_negative.sum()))
print("Rows with all six amounts equal to zero:", int(all_zero.sum()))

negative_impact = pd.DataFrame({
    "negative_sum": df[MONEY].where(df[MONEY].lt(0), 0).sum(),
    "net_sum": df[MONEY].sum(),
}).reset_index().rename(columns={"index": "attribute"})
display(negative_impact.round(2))

**Decision:** keep zeros and negative amounts as reported. Their interpretation is not fully confirmed; changing them to positive values or imputing them would alter the source data. Review their effects separately in later analyses.

## 6. Cleaning: what changed?

In [ ]:
cleaning_changes = []
for col in CATEGORIES:
    original = df[f"{col}_raw"].astype("string")
    cleaned = df[col].astype("string")
    changed = original.ne(cleaned).fillna(False)
    cleaning_changes.append({"attribute": col, "changed_cells": int(changed.sum())})
display(pd.DataFrame(cleaning_changes))
print("Rows retained:", f"{len(df):,}")
print("Negative rows marked for review:", int(df["review_negative_monetary"].sum()))

Whitespace at the start or end of categorical labels was trimmed. Unclassified sectors, zeros and negative amounts were not deleted. The original category values remain available in `_raw` columns for comparison.

## 7. Final validation

In [ ]:
status_path = root / "data/processed/common_validation.json"
if not status_path.exists():
    raise FileNotFoundError("Validation report not found. Run the processing pipeline first.")
status = json.loads(status_path.read_text(encoding="utf-8"))
print("Mandatory checks:", status.get("mandatory_status"))
print("Records in validation report:", status.get("records"))
print("Checks:", status.get("checks"))
print("Domain status:", status.get("domain_status"))
assert status.get("mandatory_status") == "PASS", "Mandatory validation did not pass."
assert status.get("records") == len(df), "Row count differs from validation report."
assert df[MONEY].notna().all().all(), "Unexpected null in monetary columns."
print("Dataset ready for analytical pipelines (with documented domain limitations).")

### Conclusion

The common dataset contains the cleaned and integrated reporting groups. It can be used as input for aggregation, sampling and feature engineering.